# Gold Price ML Starter Notebook

Welcome! This notebook walks you through a complete beginner-friendly pipeline for gold price prediction:

1. **Load** the unified CSV produced by the Yahoo Finance download script
2. **Explore** the data (EDA: head, info, missing values, basic plots)
3. **Engineer** features and a target variable (gold price / up-down label)
4. **Train** a simple model (Linear Regression + Decision Tree)
5. **Visualize** predictions vs. actuals

> **Tip:** Run each cell from top to bottom using *Shift+Enter*.

## 0. Install / import libraries

Make sure you have the required packages.  
If any are missing, uncomment the `pip install` line and run it once.

In [ ]:
# Uncomment the next line if you need to install missing packages:
# !pip install pandas numpy matplotlib seaborn scikit-learn

import os
import warnings
warnings.filterwarnings('ignore')  # keep output tidy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report
)

# Make plots look nice
sns.set_theme(style='whitegrid')
%matplotlib inline

print('All libraries loaded successfully!')

---
## 1. Load the Data

The Yahoo Finance download script saves a unified CSV to `data/yahoo_market_data.csv`.
Columns look like `gold_Open`, `gold_Close`, `sp500_Close`, `dxy_Close`, etc.

The `Date` column is used as the index.

In [ ]:
# ----------------------------------------------------------------
# Path to the CSV produced by scripts/download_market_data.py
# Adjust if your file lives somewhere else.
# ----------------------------------------------------------------
CSV_PATH = os.path.join('..', 'data', 'yahoo_market_data.csv')

# If the real file doesn't exist yet, we create a small synthetic
# dataset so you can still run through the whole notebook.
if not os.path.exists(CSV_PATH):
    print(f"'{CSV_PATH}' not found – generating synthetic data for demonstration.")
    np.random.seed(42)
    n = 500
    dates = pd.date_range(start='2020-01-01', periods=n, freq='B')  # business days
    gold_close = 1700 + np.cumsum(np.random.randn(n) * 5)
    sp500_close = 3200 + np.cumsum(np.random.randn(n) * 15)
    dxy_close = 92 + np.cumsum(np.random.randn(n) * 0.3)
    volume = np.random.randint(100_000, 500_000, size=n).astype(float)
    df_demo = pd.DataFrame({
        'gold_Open':  gold_close - np.abs(np.random.randn(n)),
        'gold_High':  gold_close + np.abs(np.random.randn(n) * 3),
        'gold_Low':   gold_close - np.abs(np.random.randn(n) * 3),
        'gold_Close': gold_close,
        'gold_Volume': volume,
        'sp500_Close': sp500_close,
        'dxy_Close':  dxy_close,
    }, index=dates)
    df_demo.index.name = 'Date'
    df = df_demo
else:
    # parse_dates=True converts the Date column to datetime automatically
    df = pd.read_csv(CSV_PATH, parse_dates=['Date'], index_col='Date')
    print(f"Loaded '{CSV_PATH}' – {len(df):,} rows.")

df.head()

---
## 2. Exploratory Data Analysis (EDA)

Before building a model, always explore the raw data.  
Ask yourself:
- What does the data look like?
- Are there missing values?
- What are the ranges / scales of each column?

In [ ]:
# ----- 2a. Shape and column names -----
print('Shape (rows, columns):', df.shape)
print('\nColumn names:')
print(df.columns.tolist())

In [ ]:
# ----- 2b. Data types and non-null counts -----
# 'object' dtype usually means text; we want floats for numeric columns.
df.info()

In [ ]:
# ----- 2c. Basic statistics -----
df.describe().round(2)

In [ ]:
# ----- 2d. Missing value (NaN) analysis -----
nan_counts = df.isna().sum()
nan_pct = (nan_counts / len(df) * 100).round(2)

nan_report = pd.DataFrame({'NaN count': nan_counts, 'NaN %': nan_pct})
print('Missing values per column:')
print(nan_report[nan_report['NaN count'] > 0] if nan_counts.sum() > 0 else 'No missing values!')

In [ ]:
# ----- 2e. Gold closing price over time -----
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df['gold_Close'], color='goldenrod', linewidth=1.5, label='Gold Close')
ax.set_title('Gold Closing Price Over Time', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ----- 2f. Distribution of daily gold returns -----
# Daily return = (today - yesterday) / yesterday  (expressed as %)
df['gold_return'] = df['gold_Close'].pct_change() * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['gold_return'].dropna(), bins=50, color='goldenrod', edgecolor='white')
ax.axvline(0, color='black', linewidth=1, linestyle='--')
ax.set_title('Distribution of Daily Gold Returns (%)', fontsize=14)
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# ----- 2g. Correlation heatmap -----
# Pick only numeric columns that exist in the dataset
numeric_cols = df.select_dtypes(include='number').columns.tolist()

fig, ax = plt.subplots(figsize=(9, 6))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # mask upper triangle to show only lower
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, ax=ax, linewidths=0.5
)
ax.set_title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Feature Engineering

Raw prices aren't great features on their own because they are non-stationary.
We create *engineered* features that better capture patterns:

| Feature | Meaning |
|---|---|
| `ma_5` / `ma_20` | 5-day and 20-day moving average of gold close |
| `return_1d` | Yesterday's gold daily return |
| `return_5d` | 5-day cumulative return |
| `volatility_5d` | Rolling 5-day standard deviation of returns |
| `sp500_return` | Previous day S&P 500 return (gold often moves opposite) |
| `dxy_return` | Previous day US Dollar Index return |

We also define **two target variables**:
- **Regression target**: next day's gold close price (`target_price`)
- **Classification target**: will gold go **up** tomorrow? (`target_up`, 1 = up, 0 = down)

In [ ]:
# ---- Start from a clean copy of the raw data ----
data = df.copy()

# --- Moving averages ---
data['ma_5']  = data['gold_Close'].rolling(5).mean()
data['ma_20'] = data['gold_Close'].rolling(20).mean()

# --- Lagged gold returns (shifted by 1 day so we don't use future data) ---
data['return_1d'] = data['gold_Close'].pct_change().shift(1)
data['return_5d'] = data['gold_Close'].pct_change(5).shift(1)

# --- Rolling volatility ---
data['volatility_5d'] = data['gold_Close'].pct_change().rolling(5).std().shift(1)

# --- Other asset returns (lagged by 1 day) ---
if 'sp500_Close' in data.columns:
    data['sp500_return'] = data['sp500_Close'].pct_change().shift(1)
if 'dxy_Close' in data.columns:
    data['dxy_return'] = data['dxy_Close'].pct_change().shift(1)

# ---- Target variables ----
# Regression: gold close price ONE DAY in the future
data['target_price'] = data['gold_Close'].shift(-1)

# Classification: 1 if tomorrow's close > today's close, else 0
data['target_up'] = (data['gold_Close'].shift(-1) > data['gold_Close']).astype(int)

# ---- Drop rows with NaN (caused by rolling windows and shifts) ----
data.dropna(inplace=True)

print(f'Dataset after feature engineering: {data.shape}')
data.head()

---
## 4. Train / Test Split

For time-series data we **never shuffle** rows — we split chronologically so the model
trains on older data and is tested on newer data (simulating real-world use).

In [ ]:
# Select feature columns (everything we engineered, drop raw OHLCV and targets)
drop_cols = ['gold_Open', 'gold_High', 'gold_Low', 'gold_Close', 'gold_Volume',
             'gold_return', 'sp500_Close', 'dxy_Close',
             'target_price', 'target_up']
feature_cols = [c for c in data.columns if c not in drop_cols]
print('Feature columns:', feature_cols)

X = data[feature_cols]
y_price = data['target_price']   # for regression
y_up    = data['target_up']      # for classification

# 80 / 20 chronological split
split = int(len(X) * 0.80)

X_train, X_test   = X.iloc[:split], X.iloc[split:]
y_price_train, y_price_test = y_price.iloc[:split], y_price.iloc[split:]
y_up_train, y_up_test       = y_up.iloc[:split],    y_up.iloc[split:]

print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

---
## 5a. Regression Model – Linear Regression (Predict Gold Price)

Linear Regression tries to find the best straight-line relationship
between the features and the next day's gold price.

In [ ]:
# Train
lr = LinearRegression()
lr.fit(X_train, y_price_train)

# Predict on test set
y_pred_price = lr.predict(X_test)

# Evaluate
mae  = mean_absolute_error(y_price_test, y_pred_price)
rmse = mean_squared_error(y_price_test, y_pred_price) ** 0.5
r2   = r2_score(y_price_test, y_pred_price)

print('=== Linear Regression Results ===')
print(f'  MAE  (Mean Absolute Error):  {mae:.2f} USD')
print(f'  RMSE (Root Mean Sq. Error):  {rmse:.2f} USD')
print(f'  R²   (variance explained):   {r2:.4f}')
print()
print('Interpretation:')
print(f'  On average the model\'s price prediction is off by ~{mae:.1f} USD.')
print(f'  R² of {r2:.2f} means the model explains {r2*100:.1f}% of price variance.')

---
## 5b. Classification Model – Decision Tree (Predict Up / Down)

Instead of predicting the exact price, we predict the *direction*:
will gold go **up** (1) or **down** (0) tomorrow?

In [ ]:
# Train a shallow decision tree (max_depth limits overfitting)
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_up_train)

# Predict
y_pred_up = dt.predict(X_test)

# Evaluate
acc = accuracy_score(y_up_test, y_pred_up)
print('=== Decision Tree Classifier Results ===')
print(f'  Accuracy: {acc:.2%}\n')
print(classification_report(y_up_test, y_pred_up, target_names=['Down (0)', 'Up (1)']))

---
## 6. Visualize Predictions vs. Actuals

Numbers alone don't tell the full story — always plot your predictions!

In [ ]:
# ----- 6a. Regression: Predicted vs Actual gold price (line plot) -----
fig, ax = plt.subplots(figsize=(13, 4))

test_index = X_test.index

ax.plot(test_index, y_price_test.values,  color='goldenrod',  linewidth=1.5, label='Actual Price')
ax.plot(test_index, y_pred_price,          color='steelblue',  linewidth=1.5,
        linestyle='--', label='Predicted Price')

ax.set_title('Linear Regression – Predicted vs Actual Gold Price (Test Set)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Gold Close Price (USD)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ----- 6b. Regression: Scatter plot (perfect model = diagonal line) -----
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_price_test, y_pred_price, alpha=0.4, color='steelblue', edgecolors='white', s=30)

# Draw the 'perfect prediction' diagonal
lo, hi = y_price_test.min(), y_price_test.max()
ax.plot([lo, hi], [lo, hi], color='red', linewidth=1.5, linestyle='--', label='Perfect fit')

ax.set_title('Actual vs. Predicted Price', fontsize=13)
ax.set_xlabel('Actual Gold Price (USD)')
ax.set_ylabel('Predicted Gold Price (USD)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ----- 6c. Classification: Confusion matrix -----
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(y_up_test, y_pred_up)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Down', 'Up'])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Decision Tree – Confusion Matrix', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ----- 6d. Feature importance (Decision Tree) -----
importances = pd.Series(dt.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Decision Tree – Feature Importances', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

---
## 7. Summary & Next Steps

### What we did
| Step | Details |
|---|---|
| Loaded data | `data/yahoo_market_data.csv` (or synthetic fallback) |
| EDA | Head, info, NaN stats, price chart, return distribution, correlation heatmap |
| Feature engineering | Moving averages, lagged returns, volatility, cross-asset returns |
| Regression | Linear Regression → next day gold close price |
| Classification | Decision Tree → next day direction (Up / Down) |
| Visualization | Time-series, scatter, confusion matrix, feature importance |

### Ideas to improve the model
- **More features**: RSI, MACD, Bollinger Bands, crude oil price, VIX
- **Better models**: Random Forest, Gradient Boosting (XGBoost / LightGBM), LSTM
- **Hyperparameter tuning**: `GridSearchCV` or `Optuna`
- **Cross-validation**: Walk-forward validation for time-series
- **Risk management**: Position sizing, stop-loss rules

Happy modelling! 🥇